### Creating KG Layers

In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

In [2]:
host_layer = pd.read_csv('host_layer_data.csv') #L1
amr_layer = pd.read_csv('amr_layer_data.csv') #L3
drugclass_gene_addon = pd.read_csv('drugclass_gene_data.csv') #L2
resistance_gene_data = pd.read_csv('resistance_gene_data_18_07.csv')

Creating infection-pathogen relation

In [3]:
pathogen_disease_infection = pd.DataFrame({'x_name':['Klebsiella pneumoniae','Klebsiella pneumoniae', 'Klebsiella pneumoniae','Streptococcus pneumoniae', 'Streptococcus pneumoniae', 'Escherichia coli','Escherichia coli', 'Acinetobacter baumannii','Acinetobacter baumannii', 'Mycobacterium tuberculosis',
                                              'Enterococcus faecalis',  'Enterococcus faecalis',  'Enterococcus faecalis', 'Staphylococcus aureus','Staphylococcus aureus','Staphylococcus aureus','Staphylococcus aureus', 'Pseudomonas aeruginosa','Pseudomonas aeruginosa','Pseudomonas aeruginosa','Burkholderia pseudomallei','Porphyromonas gingivalis'], 
                                              'y_name':['pneumonia','urinary tract infection (disease)','cellulitis (disease)','pneumonia','Bacteremia','urinary tract infection (disease)','Bacteremia','pneumonia','Bacteremia','tuberculosis','Bacteremia','urinary tract infection (disease)','Penetrating foot ulcers','cellulitis (disease)','Penetrating foot ulcers',
                                                        'folliculitis','osteomyelitis (disease)','pneumonia','Penetrating foot ulcers','otitis externa','melioidosis','periodontitis']})

In [4]:
pathogen_disease_infection['x_type'] = ['bacteria']*len(pathogen_disease_infection)
pathogen_disease_infection['y_type'] = ['disease']*len(pathogen_disease_infection)
pathogen_disease_infection['relation'] = ['pathogen_disease']*len(pathogen_disease_infection)
pathogen_disease_infection['display_relation'] = ['infectious_agent_for']*len(pathogen_disease_infection)

In [5]:
#using pathogen_dict from the AMR cleaning notebook for ease of ID assignment
pathogen_dict = {'Klebsiella pneumoniae':'B0', 'Streptococcus pneumoniae':'B1', 'Escherichia coli':'B2', 'Acinetobacter baumannii':'B3', 
                 'Mycobacterium tuberculosis':'B4', 'Enterococcus faecalis':'B5', 'Staphylococcus aureus':'B6', 'Pseudomonas aeruginosa':'B7','Burkholderia pseudomallei':'B8','Porphyromonas gingivalis':'B9'}

In [6]:
#Filling the pathogen_infection dataframe with created IDs
def id_fxn(row):
  
    #For pathogen
    row['x_index'] = pathogen_dict[row['x_name']]

    #For disease
    row['y_index'] = host_layer[host_layer['y_name'] == row['y_name']].iloc[0,5]
 
    return row

pathogen_disease_infection = pathogen_disease_infection.apply(lambda x: id_fxn(x), axis = 1)

In [7]:
pathogen_disease_infection = pathogen_disease_infection[['relation','display_relation','x_index','x_type','x_name','y_index','y_type','y_name']] #L2

In [8]:
pathogen_disease_infection.head(3)

,relation,display_relation,x_index,x_type,x_name,y_index,y_type,y_name
0,pathogen_disease,infectious_agent_for,B0,bacteria,Klebsiella pneumoniae,35552,disease,pneumonia
1,pathogen_disease,infectious_agent_for,B0,bacteria,Klebsiella pneumoniae,36322,disease,urinary tract infection (disease)
2,pathogen_disease,infectious_agent_for,B0,bacteria,Klebsiella pneumoniae,38022,disease,cellulitis (disease)


### Including some missing imporant links & curating causal layer

First -- Missing links between 'recurrent infections' and main infections. This also then connects T2DM (aside the connection through immunodeficiency) 
to recurrent infections that will connect to ARGs

In [9]:
primekg_nodes = pd.read_csv('dataverse_files/nodes.csv')
primekg = pd.read_csv('dataverse_files/kg.csv')

/tmp/ipykernel_4926/3807702634.py:2: DtypeWarning: Columns (3,8) have mixed types. Specify dtype option on import or set low_memory=False.
  primekg = pd.read_csv('dataverse_files/kg.csv')


Add-on 1

In [10]:
recurrent_hierarchy = ['Recurrent bacterial infections', 'Recurrent staphylococcal infections', 'Recurrent ear infections', 'Recurrent mycobacterial infections', 
                       'Recurrent streptococcal infections', 'Recurrent gram-negative bacterial infections', 'Recurrent respiratory infections']

In [11]:
recurrent_higher_level_addon = primekg[(primekg['y_name'].isin(recurrent_hierarchy)) & (primekg['x_type'] == 'effect/phenotype')][['relation','display_relation','x_index','x_type','x_name','y_index','y_type','y_name']] #L1

Add-on 2

In [12]:
#adding DFI connected to poor wound healing 
recurrent_keys = ['Recurrent Klebsiella infections', 'Recurrent Klebsiella infections', 'Recurrent Klebsiella infections',  'Recurrent streptococcus pneumoniae infections',  'Recurrent streptococcus pneumoniae infections',
                  'Recurrent mycobacterial infections', 'Recurrent Staphylococcus aureus infections', 'Recurrent Staphylococcus aureus infections', 'Recurrent Staphylococcus aureus infections', 'Recurrent Staphylococcus aureus infections', 
                  'Recurrent bacterial skin infections',  'Recurrent bacterial skin infections',  'Recurrent bacterial skin infections', 'Recurrent ear infections', 'Recurrent lower respiratory tract infections', 'Poor wound healing']

rec_associated_infections = ['pneumonia','urinary tract infection (disease)','cellulitis (disease)','pneumonia','Bacteremia','tuberculosis', 'cellulitis (disease)','Penetrating foot ulcers',
                        'folliculitis','osteomyelitis (disease)', 'cellulitis (disease)', 'folliculitis','Penetrating foot ulcers','otitis externa', 'pneumonia', 'Penetrating foot ulcers']

In [13]:
recurrent_to_infection = {'relation': ['disease_phenotype_positive']*len(recurrent_keys) ,'display_relation':['phenotype present']*len(recurrent_keys), 'x_index':np.zeros(len(recurrent_keys)),'x_type':['disease']*len(recurrent_keys),'x_name':rec_associated_infections,
                          'y_index':np.zeros(len(recurrent_keys)),'y_type': ['effect/phenotype']*len(recurrent_keys), 'y_name':recurrent_keys}

In [14]:
#Filling dataframe with created IDs with our usual function structure
def id_fxn(row):
  
    #For recurrent
    row['y_index'] = primekg_nodes[primekg_nodes['node_name'] == row['y_name']].iloc[0,0]

    #For disease, using host layer to account for changes like reconcilations made
    row['x_index'] = host_layer[host_layer['y_name'] == row['x_name']].iloc[0,5]
 
    return row

recurrent_to_infection_df = pd.DataFrame(recurrent_to_infection)

recurrent_to_infection_df = recurrent_to_infection_df.apply(lambda x: id_fxn(x), axis = 1) #L1

Second -- Curating connecting relations

Add-on 3

In [15]:
#Connecting some of the useful immune dysfunction nodes directly to immunodeficiency as they are (not surprisingly) indirectly connected. 
for_immuno_update = host_layer[(host_layer['x_name'] == 'Immunodeficiency') & (host_layer['y_type'] == 'immune_effect/phenotype')].head(6).copy()

In [16]:
for_immuno_update['y_name'] = ['Neutropenia', 'Decreased serum complement C3', 'Impaired T cell function', 'Defective B cell differentiation', 'Impaired neutrophil bactericidal activity', 'Cytokine storm']

In [17]:
#Filling dataframe with created IDs with our usual function structure
def id_fxn(row):
  
    #For immuno_update
    row['y_index'] = primekg_nodes[primekg_nodes['node_name'] == row['y_name']].iloc[0,0]
 
    return row

for_immuno_update = for_immuno_update.apply(lambda x: id_fxn(x), axis = 1) #L1

Actual curation below

Add-on 4

In [18]:
immune_dys = ['Humoral immunodeficiency','Humoral immunodeficiency',
              'Decreased circulating antibody level', 'Decreased circulating antibody level','Decreased circulating antibody level','Decreased circulating antibody level','Decreased circulating antibody level','Decreased circulating antibody level',
              'Decreased proportion of CD4-positive T cells','Decreased proportion of CD4-positive T cells','Decreased proportion of CD4-positive T cells','Decreased proportion of CD4-positive T cells',
              'Decreased proportion of memory B cells','Decreased proportion of memory B cells','Decreased proportion of memory B cells','Decreased proportion of memory B cells',
              'Neutropenia', 'Neutropenia', 'Neutropenia', 'Neutropenia', 
              'Decreased serum complement C3', 'Decreased serum complement C3',
              'Impaired T cell function', 'Impaired T cell function', 'Impaired T cell function', 'Impaired T cell function', 'Impaired T cell function', 'Impaired T cell function', 
              'Defective B cell differentiation', 'Defective B cell differentiation','Defective B cell differentiation','Defective B cell differentiation',
              'Impaired neutrophil bactericidal activity', 'Impaired neutrophil bactericidal activity','Impaired neutrophil bactericidal activity','Impaired neutrophil bactericidal activity','Impaired neutrophil bactericidal activity',
              'Cytokine storm', 'Cytokine storm']

pathogens_to_immune = ['Streptococcus pneumoniae','Staphylococcus aureus',
                       'Klebsiella pneumoniae','Streptococcus pneumoniae','Pseudomonas aeruginosa','Acinetobacter baumannii','Staphylococcus aureus','Mycobacterium tuberculosis',
                       'Klebsiella pneumoniae','Streptococcus pneumoniae','Pseudomonas aeruginosa','Acinetobacter baumannii',
                       'Klebsiella pneumoniae','Streptococcus pneumoniae','Pseudomonas aeruginosa','Acinetobacter baumannii',
                       'Klebsiella pneumoniae','Pseudomonas aeruginosa','Staphylococcus aureus','Escherichia coli',
                       'Streptococcus pneumoniae','Staphylococcus aureus',
                       'Streptococcus pneumoniae','Staphylococcus aureus','Klebsiella pneumoniae','Escherichia coli','Acinetobacter baumannii','Enterococcus faecalis',
                       'Streptococcus pneumoniae','Staphylococcus aureus','Klebsiella pneumoniae','Acinetobacter baumannii',
                       'Streptococcus pneumoniae','Staphylococcus aureus','Klebsiella pneumoniae','Acinetobacter baumannii','Pseudomonas aeruginosa',
                       'Streptococcus pneumoniae','Porphyromonas gingivalis']

In [19]:
curated_immune_to_pathogen = {'relation': ['immune_dys_pathogen']*len(immune_dys) ,'display_relation':['increases_susceptibility_to']*len(immune_dys), 'x_index':np.zeros(len(immune_dys)),'x_type':['immune_effect/phenotype']*len(immune_dys),'x_name':immune_dys,
                          'y_index':np.zeros(len(immune_dys)),'y_type': ['bacteria']*len(immune_dys), 'y_name':pathogens_to_immune}

In [20]:
#Filling dataframe with created IDs with our usual function structure
def id_fxn(row):
  
    #For immune_dysfunction
    row['x_index'] = primekg_nodes[primekg_nodes['node_name'] == row['x_name']].iloc[0,0]

    #For pathogen
    row['y_index'] = pathogen_dict[row['y_name']]
 
    return row

curated_immune_to_pathogen_df = pd.DataFrame(curated_immune_to_pathogen)

curated_immune_to_pathogen_df = curated_immune_to_pathogen_df.apply(lambda x: id_fxn(x), axis = 1) #L2

In [21]:
#semi_kg_data = pd.concat([host_layer,amr_layer,drugclass_gene_addon,pathogen_disease_infection, recurrent_to_infection_df, for_immuno_update, curated_immune_to_pathogen_df, recurrent_higher_level_addon], ignore_index= True)

Final loose ends

Recurrent bacterial infections --> increased antibiotic exposure 

Hyperglycemia --> Increased risk of pathogen mutation --> increased antibiotic exposure. 

[Calls for some amount of hard coding]

Add-on 5

In [22]:
extra_connect = {'relation':['phenotype_phenotype']*3,'display_relation':['parent-child']*3,'x_index':[23984,25526,140001],'x_type':['effect/phenotype']*3,'x_name':['Recurrent bacterial infections', 'Hyperglycemia', 'Increased risk of pathogen mutation'],'y_index':[140000, 140001, 140000],'y_type':['effect/phenotype']*3,'y_name':['Increased antibiotic exposure', 'Increased risk of pathogen mutation', 'Increased antibiotic exposure']}

In [23]:
extra_connect_df = pd.DataFrame(extra_connect) #L1

In [24]:
full_L1 = pd.concat([host_layer, recurrent_higher_level_addon, recurrent_to_infection_df, for_immuno_update, extra_connect_df], ignore_index=True)

In [25]:
l1_nodes = pd.concat([full_L1[['x_name','x_type','x_index']].rename(columns={'x_index':'node_index','x_type':'node_type','x_name':'node_name'}),full_L1[['y_name','y_type','y_index']].rename(columns={'y_index':'node_index','y_type':'node_type','y_name':'node_name'})],ignore_index=True)
groupings = l1_nodes.agg(frozenset,axis=1)
l1_nodes = l1_nodes.groupby(groupings).first().reset_index(drop=True)

Increased antibiotic exposure --> ARGs

Add-on 6

In [26]:
amr_selection1 = pd.DataFrame({'relation':['phenotype_resistance']*len(resistance_gene_data),'display_relation':['increased_selection_for']*len(resistance_gene_data),'x_index':[140000]*len(resistance_gene_data),'x_type':['effect/phenotype']*len(resistance_gene_data),'x_name':['Increased antibiotic exposure']*len(resistance_gene_data), 'y_type': ['resistance_gene']*len(resistance_gene_data)})

In [27]:
amr_selection = pd.concat([amr_selection1,resistance_gene_data.rename(columns={'ARO_Accession':'y_index','Resistance_Gene':'y_name'})], axis = 1) #L2

In [28]:
full_L2 = pd.concat([drugclass_gene_addon,pathogen_disease_infection,curated_immune_to_pathogen_df,amr_selection], ignore_index=True)

In [29]:
l2_nodes = pd.concat([full_L2[['x_name','x_type','x_index']].rename(columns={'x_index':'node_index','x_type':'node_type','x_name':'node_name'}),full_L2[['y_name','y_type','y_index']].rename(columns={'y_index':'node_index','y_type':'node_type','y_name':'node_name'})],ignore_index=True)
groupings = l2_nodes.agg(frozenset,axis=1)
l2_nodes = l2_nodes.groupby(groupings).first().reset_index(drop=True)

In [30]:
full_L3 = amr_layer

In [31]:
l3_nodes = pd.concat([full_L3[['x_name','x_type','x_index']].rename(columns={'x_index':'node_index','x_type':'node_type','x_name':'node_name'}),full_L3[['y_name','y_type','y_index']].rename(columns={'y_index':'node_index','y_type':'node_type','y_name':'node_name'})],ignore_index=True)
groupings = l3_nodes.agg(frozenset,axis=1)
l3_nodes = l3_nodes.groupby(groupings).first().reset_index(drop=True)

Finally compiling all together, may not be the last time :D

In [32]:
whole_kg_data = pd.concat([full_L1,full_L2,full_L3], ignore_index= True)

In [33]:
groupings = whole_kg_data.agg(frozenset, axis=1)
whole_kg_data = whole_kg_data.groupby(groupings).first().reset_index(drop=True)

Creating an all nodes dataframe from wholekg dataframe to include all changes and new nodes

In [34]:
whole_nodes = pd.concat([whole_kg_data[['x_name','x_type','x_index']].rename(columns={'x_index':'node_index','x_type':'node_type','x_name':'node_name'}),whole_kg_data[['y_name','y_type','y_index']].rename(columns={'y_index':'node_index','y_type':'node_type','y_name':'node_name'})],ignore_index=True)
groupings = whole_nodes.agg(frozenset,axis=1)
whole_nodes = whole_nodes.groupby(groupings).first().reset_index(drop=True)

In [35]:
len(whole_nodes)

12191

In [36]:
len(whole_nodes['node_index'].unique()) == len(whole_nodes)

True

KG creation

In [37]:
#Function for creating list of nodes and edges based on networkx documentation for adding nodes and edges

def create_node_list(nodesdf):
    node_list = []
    for i in range(len(nodesdf)):
        node_list.append((nodesdf.loc[i,'node_name'],{'node_index':nodesdf.loc[i,'node_index'],'node_type':nodesdf.loc[i,'node_type']}))
    
    return node_list

def create_relation_list(edgesdf):
    edge_list = []

    for i in range(len(edgesdf)):
        edge_list.append((edgesdf.loc[i,'x_name'],edgesdf.loc[i,'y_name'],{'relation':edgesdf.loc[i,'relation'],'display_relation':edgesdf.loc[i,'display_relation']}))
    
    return edge_list

In [38]:
host_layerkg = nx.DiGraph()
host_layerkg.add_nodes_from(create_node_list(l1_nodes))
host_layerkg.add_edges_from(create_relation_list(full_L1))

In [39]:
causal_layerkg = nx.DiGraph()
causal_layerkg.add_nodes_from(create_node_list(l2_nodes))
causal_layerkg.add_edges_from(create_relation_list(full_L2))

In [40]:
amr_layerkg = nx.DiGraph()
amr_layerkg.add_nodes_from(create_node_list(l3_nodes))
amr_layerkg.add_edges_from(create_relation_list(full_L3))

In [41]:
amr_layerkg.nodes()

NodeView(('Escherichia coli ompF with mutation conferring resistance to beta-lactam antibiotics', 'SHV-14', 'OXA-207', 'QnrB7', 'dfrA23', 'IMP-34', 'mdtB', 'CTX-M-68', 'CMY-10', 'TEM-26', 'mecC', 'mfpA', 'CMY-24', 'lnuB', 'MexC', 'QnrA3', 'catA1', 'PC1 beta-lactamase (blaZ)', 'GES-11', "AAC(6')-Ij", 'vanR gene in vanA cluster', 'OXA-28', 'QnrB32', 'PDC-3', 'CTX-M-8', 'TEM-171', 'CTX-M-55', 'NDM-1', 'SHV-49', 'CTX-M-58', 'GES-14', 'SHV-61', 'OXA-232', 'adeS', 'arlS', "AAC(6')-Iaf", 'PDC-8', 'mdtF', 'arr-8', 'TEM-198', 'vanH gene in vanO cluster', 'VIM-34', 'VEB-5', 'fosA5', 'mgrA', 'adeF', 'cpxA', 'QnrB10', 'catB3', 'aadA12', 'OXA-98', 'vanR gene in vanB cluster', "AAC(6')-IIb", 'aadA4', 'APH(4)-Ia', 'GES-13', 'OXA-198', 'CTX-M-99', 'TolC', 'TEM-68', 'TEM-163', 'ugd', 'CTX-M-14', 'CMY-44', 'CTX-M-102', 'CMY-58', "APH(3')-Ib", 'FosA3', 'rmtB', 'SHV-157', 'mepR', 'ErmC', 'dfrA19', 'cAMP-RP', 'DHA-1', 'adeB', 'OXA-244', 'PER-7', 'IMP-16', 'dfrB1', 'OXA-4', 'Mycobacterium tuberculosis pncA 

In [42]:
wholekg = nx.DiGraph()
wholekg.add_nodes_from(create_node_list(whole_nodes))
wholekg.add_edges_from(create_relation_list(whole_kg_data))

In [43]:
wholekg.number_of_nodes()

12190

In [44]:
wholekg.number_of_edges()

30803

Note that 'Maturation of protein 3a' is genuinely duplicated in PrimeKG hence the drop from 12191 to 12190

In [45]:
#For plot on Gephi
nx.write_gexf(wholekg,'draft2_KG.gexf')

### KG Analysis

Credit to MW

In [46]:
TOP_N = 15               # how many top hub nodes to report/plot

In [47]:
poi_card_df = pd.read_csv('cleaned_raw_dataAMRpoi.csv')

Centrality entities

In [48]:
def degree_centrality_and_hubs(G):
    degree_c = nx.degree_centrality(G)
    betw_c = nx.betweenness_centrality(G, weight="weight")

    result = pd.DataFrame({
        "node": list(degree_c.keys()),
        "degree_centrality": list(degree_c.values()),
        "node_type": [G.nodes[n]["node_type"] for n in degree_c.keys()],
    }).sort_values("degree_centrality", ascending=False)

    #focused_result = result[result['node_type'] == 'biological_process']

    result.to_csv(f"network_analysis_results/'{G}'_d_centrality.csv", index=False)

    print(f"\n=== Top {TOP_N} nodes by degree centrality ===")
    print(result.head(TOP_N).to_string(index=False))

    return result

In [49]:
def betweeness_centrality(G):
    betw_c = nx.betweenness_centrality(G, weight="weight")

    result = pd.DataFrame({
        "node": list(betw_c.keys()),
        "betweeness_centrality": list(betw_c.values()),
        "node_type": [G.nodes[n]["node_type"] for n in betw_c.keys()],
    }).sort_values("betweeness_centrality", ascending=False)

    #focused_result = result[result['node_type'] == 'biological_process']

    result.to_csv(f"network_analysis_results/'{G}'_b_centrality.csv", index=False)

    print(f"\n=== Top {TOP_N} nodes by betweeness centrality ===")
    print(result.head(TOP_N).to_string(index=False))

    return result

In [50]:
whole_analysis = degree_centrality_and_hubs(wholekg)


=== Top 15 nodes by degree centrality ===
                            node  degree_centrality            node_type
   Increased antibiotic exposure           0.142423     effect/phenotype
                            ETS1           0.125195         gene/protein
         antibiotic inactivation           0.103044 resistance_mechanism
                   cephalosporin           0.089835           drug_class
                      carbapenem           0.062351           drug_class
                            EGFR           0.061941         gene/protein
           Klebsiella pneumoniae           0.055050             bacteria
          Pseudomonas aeruginosa           0.054311             bacteria
        hepatocellular carcinoma           0.053819              disease
                           penam           0.053573           drug_class
X-linked intellectual disability           0.045697              disease
         Acinetobacter baumannii           0.045205             bacteria
        

In [51]:
#amr_b_analysis = betweeness_centrality(amr_layerkg)

More analysis

In [52]:
def quantify_coverage_bias(card_df):
    """
    Produces the coverage table required by the Week 1/2 feedback:
    rows per pathogen, distinct AMR genes, distinct mechanisms, distinct drug classes.
    """
    card_df = card_df.copy()

    rows = []
    for pathogen, group in card_df.groupby("Pathogen"):
        rows.append({
            "pathogen": pathogen,
            "n_rows": len(group),
            "n_amr_genes": group["Resistance_Gene"].nunique() if "Resistance_Gene" in group.columns else None,
            "n_amr_mechanisms": group["Resistance_Mechanism"].nunique() if "Resistance_Mechanism" in group.columns else None,
            "n_drug_class": group["Drug_Class"].nunique() if "Drug_Class" in group.columns else None
        })

    coverage_df = pd.DataFrame(rows).sort_values("n_amr_genes", ascending=False)
    coverage_df.to_csv(r"card_coverage_bias.csv", index=False)

    print("\n=== CARD coverage by pathogen (potential bias — see Week 2 feedback) ===")
    print(coverage_df.to_string(index=False))
    print(
        "\nNote: low row counts for e.g. M. tuberculosis / S. pneumoniae likely reflect "
        "chromosomal-mutation-based resistance being under-captured by this CARD model, "
        "NOT lower biological relevance. Report this alongside any centrality results below."
    )
    return coverage_df


In [53]:
amr_coverage_df = quantify_coverage_bias(poi_card_df)


=== CARD coverage by pathogen (potential bias — see Week 2 feedback) ===
                  pathogen  n_rows  n_amr_genes  n_amr_mechanisms  n_drug_class
     Klebsiella pneumoniae    1561          627                 7            34
    Pseudomonas aeruginosa    1804          624                 7            30
   Acinetobacter baumannii    1069          516                 6            27
          Escherichia coli    1106          474                 6            34
     Staphylococcus aureus     247          130                 6            30
     Enterococcus faecalis     152           98                 5            23
Mycobacterium tuberculosis     111           45                 5            30
  Streptococcus pneumoniae      68           35                 5            20
 Burkholderia pseudomallei      25           11                 4            12
  Porphyromonas gingivalis       4            4                 2             3

Note: low row counts for e.g. M. tuberculosis

In [54]:
def shared_mechanisms_matrix(card_df):
    """
    Pathogen x mechanism/category binary matrix — lets you see, and later plot,
    which mechanisms are shared across multiple target pathogens.
    """
    rows = []
    for _, row in card_df.iterrows():
       rows.append({"pathogen": row["Pathogen"], "category": row['Resistance_Mechanism']})

    long_df = pd.DataFrame(rows).drop_duplicates()
    matrix = pd.crosstab(long_df["pathogen"], long_df["category"])
    matrix.to_csv(r"pathogen_mechanism_matrix.csv")

    # mechanisms shared by >= 2 pathogens are the most interesting for the "hub mechanism" story
    shared = matrix.loc[:, (matrix > 0).sum(axis=0) >= 2]
    print(f"\n=== Mechanisms/categories shared by >=2 target pathogens: {shared.shape[1]} found ===")
    print(list(shared.columns)[:20], "..." if shared.shape[1] > 20 else "")

    return matrix

In [55]:
shared_mech_matrix = shared_mechanisms_matrix(poi_card_df)


=== Mechanisms/categories shared by >=2 target pathogens: 7 found ===
['  reduced permeability to antibiotic', ' antibiotic efflux', ' antibiotic inactivation', ' antibiotic target alteration', ' antibiotic target protection', ' antibiotic target replacement', ' resistance by absence'] 


In [56]:
shared_mech_matrix

category,reduced permeability to antibiotic,antibiotic efflux,antibiotic inactivation,antibiotic target alteration,antibiotic target protection,antibiotic target replacement,resistance by absence
pathogen,,,,,,,
Acinetobacter baumannii,1,1,1,1,1,1,0
Burkholderia pseudomallei,1,1,1,1,0,0,0
Enterococcus faecalis,0,1,1,1,1,1,0
Escherichia coli,1,1,1,1,1,1,0
Klebsiella pneumoniae,1,1,1,1,1,1,1
Mycobacterium tuberculosis,0,1,1,1,1,1,0
Porphyromonas gingivalis,0,0,0,1,1,0,0
Pseudomonas aeruginosa,1,1,1,1,1,1,1
Staphylococcus aureus,1,1,1,1,1,1,0


In [57]:
shared_mech_matrix.columns

Index(['  reduced permeability to antibiotic', ' antibiotic efflux',
       ' antibiotic inactivation', ' antibiotic target alteration',
       ' antibiotic target protection', ' antibiotic target replacement',
       ' resistance by absence'],
      dtype='object', name='category')

In [58]:
def immune_infection_overlap(result_df, keyword_list=None):
    """
    Quantifies overlap between the T2D neighbourhood and infection/immune-related nodes,
    using the same keyword approach discussed for 'immune_effect/phenotype' —
    document the keyword list here so it stays auditable (per Week 3 feedback).
    """
    if keyword_list is None:
        keyword_list = ['lymph','NK','T cell','immune','immuno','antibody','B cell','CD','cytokines','IgA','IgG','IgM','IgE','IgD','IL-','inflammation','inflammatory','leuko','Phago','Macrophage'
            ,'neutro','monocyte','monokine','basophil','eosinophil','TNF','histio','antigen','immunodeficiency','rheumatoid factor','interleukin','interferon'
            ,'IFN','TCR', 'autoimmune', 'globulin', 'complement']
    pattern = "|".join(keyword_list)
    matches = result_df[result_df["node_name"].str.contains(pattern, case=False, na=False)]

    print(f"\n=== {len(matches)} / {len(result_df)} T2D-neighbourhood nodes match immune/infection keywords ===")
    print(f"Keyword list used (document this in your methods): {keyword_list}")
    matches.to_csv(r"host_immune_infection_overlap.csv", index=False)

    return matches

### Plot

In [59]:
def plot_shared_mechanism_heatmap(matrix, filename="fig_shared_mechanisms_heatmap.png"):
    import numpy as np
    plt.figure(figsize=(11, 6))
    plt.imshow(matrix.values, aspect="auto", cmap="viridis")
    plt.yticks(range(len(matrix.index)), matrix.index, fontsize=8)
    plt.xticks(range(len(matrix.columns)), matrix.columns, fontsize=8, rotation = 30)  
    plt.colorbar(label="")
    plt.title("Pathogen-resistance mechanism binary matrix")
    plt.tight_layout()
    plt.savefig(f"{filename}", dpi=200)
    plt.close()
    print(f"Saved {filename}")

In [60]:
plot_shared_mechanism_heatmap(shared_mech_matrix, filename="fig2_shared_mechanisms_heatmap.png")

Saved fig2_shared_mechanisms_heatmap.png
